# Пайплайн позвоночника

Две головы CNN: `spine_lay` и `spine_obj`. Ось — логистическая регрессия по углу `axis_angle`. На исследование остаётся один кадр: дубликаты схлопываются до фолдов, страта считается уже по 99 исследованиям.

15 эпох фиксированы, лучшая эпоха по валидации не выбирается. Порог каждого внешнего фолда берётся из остальных OOF-фолдов. Интервал - cluster-bootstrap по `study_id`. Это спокойнее, чем подбор порога на том же OOF-фолде, и это всё ещё не вложенный CV: предсказания соседних фолдов получены моделями, которые видели текущий фолд в обучении.


## Настройки


In [ ]:
import random

import joblib
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torchvision.transforms as T
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18

import dxa_utils as U

RANDOM_STATE = 42
IMG_SIZE = 320
EPOCHS = 15
BATCH_SIZE = 8
LR = 1e-4
WEIGHT_DECAY = 1e-4
AUG_DEGREES = 3
AUG_TRANSLATE = 0.03
AUG_BRIGHTNESS = 0.2
AUG_CONTRAST = 0.2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_COLS = ["spine_lay", "spine_obj"]
EXCEL_PATH = r"C:\Users\kiru\OneDrive\Рабочий стол\НД_для_обучения\разметка.xlsx"
LABEL_COLUMNS = [
    "n",
    "study",
    "spine_lay",
    "spine_axis",
    "spine_obj",
    "rfem_lay",
    "rfem_roi",
    "lfem_lay",
    "lfem_roi",
    "total_spine",
    "total_rfem",
    "total_lfem",
    "comment",
    "c13",
    "c14",
    "c15",
    "c16",
    "c17",
    "c18",
]
LAY_THRESHOLD = 0.7592255473136902


def set_seed(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed()
print("DEVICE:", DEVICE)
print("torch:", torch.__version__)


## Данные и фолды

Берутся кадры поясничного отдела из `metadata_train.csv`. Метки `spine_lay` и `spine_obj` подтягиваются из Excel. Строки сортируются по `study_id` и `file_path`, затем остаётся один кадр на исследование. Страта считается после этого схлопывания.

Угол оси считается по оставшемуся кадру и присоединяется отдельно. В страту он не входит.


In [ ]:
df = pd.read_csv("metadata_train.csv")
df = df[df.anatomical_region == U.SPINE_REGION].copy()
labels = pd.read_excel(EXCEL_PATH, header=None).iloc[2:].reset_index(drop=True)
labels.columns = LABEL_COLUMNS
labels["study"] = labels["study"].astype(str).str.strip()
sp = df.merge(
    labels[["study", "spine_lay", "spine_obj"]].rename(columns={"study": "study_id"}),
    on="study_id",
    how="left",
).dropna(subset=LABEL_COLS).reset_index(drop=True)
sp[LABEL_COLS] = sp[LABEL_COLS].astype(int)

sp = sp.sort_values(["study_id", "file_path"])
sp = sp.drop_duplicates("study_id").reset_index(drop=True)
strat = sp.spine_lay.astype(str) + sp.spine_obj.astype(str)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
sp["fold"] = -1
for fold, (_, val_idx) in enumerate(sgkf.split(sp, strat, groups=sp.study_id)):
    sp.loc[val_idx, "fold"] = fold

sp["axis_angle"] = sp.file_path.apply(
    lambda path: U.spine_axis_angle(pydicom.dcmread(path).pixel_array)
)
axis_y = labels[["study", "spine_axis"]].rename(columns={"study": "study_id"})
sp = sp.merge(axis_y, on="study_id", how="left")
sp["spine_axis"] = sp.spine_axis.astype(int)
sp.to_csv("spine_dataset.csv", index=False)
print(len(sp), sp[["spine_lay", "spine_obj", "spine_axis"]].sum().to_dict())


## Датасет и модель

Кадр нормализуется min-max с `+1e-6`, затем переводится в uint8. Учится только `layer4` и новая голова. Голова — два логита, `spine_lay` и `spine_obj`. Аугментация только на обучении: поворот 3°, сдвиг 0.03, яркость и контраст 0.2. Сид ставится до создания модели.


In [ ]:
class SpineDataset(Dataset):
    def __init__(self, frame, train=True):
        self.df = frame.reset_index(drop=True)
        aug = []
        if train:
            aug = [
                T.RandomAffine(AUG_DEGREES, translate=(AUG_TRANSLATE, AUG_TRANSLATE)),
                T.ColorJitter(brightness=AUG_BRIGHTNESS, contrast=AUG_CONTRAST),
            ]
        self.transform = T.Compose(
            [T.ToPILImage(), T.Resize((IMG_SIZE, IMG_SIZE)), *aug, T.ToTensor()]
        )
        self.normalize = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        arr = pydicom.dcmread(row.file_path).pixel_array.astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
        arr = (arr * 255).astype(np.uint8)
        x = self.normalize(self.transform(arr).repeat(3, 1, 1))
        y = torch.tensor(row[LABEL_COLS].values.astype(np.float32))
        return x, y


def build_model():
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith("layer4") or name.startswith("fc")
    model.fc = nn.Linear(model.fc.in_features, len(LABEL_COLS))
    return model.to(DEVICE)


def train_fold(frame, epochs=EPOCHS):
    set_seed(RANDOM_STATE)
    model = build_model()
    pos = torch.tensor(
        [
            (len(frame) - frame[col].sum()) / max(frame[col].sum(), 1)
            for col in LABEL_COLS
        ],
        dtype=torch.float32,
        device=DEVICE,
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos)
    optimizer = torch.optim.AdamW(
        [param for param in model.parameters() if param.requires_grad],
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )
    loader = DataLoader(SpineDataset(frame, True), batch_size=BATCH_SIZE, shuffle=True)
    for _ in range(epochs):
        model.train()
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
    return model


@torch.no_grad()
def pred(model, frame):
    model.eval()
    out = []
    loader = DataLoader(SpineDataset(frame, False), batch_size=BATCH_SIZE)
    for x, _ in loader:
        out.append(torch.sigmoid(model(x.to(DEVICE))).cpu().numpy())
    return np.concatenate(out)


## Обучение по фолдам

In [ ]:
oof = np.full((len(sp), 2), np.nan)
models = []
for fold in sorted(sp.fold.unique()):
    train_part = sp[sp.fold != fold]
    val_part = sp[sp.fold == fold]
    model = train_fold(train_part)
    oof[val_part.index.values] = pred(model, val_part)
    torch.save(model.state_dict(), f"spine_cnn_fold{fold}.pt")
    models.append(model)

np.save("spine_cnn_oof.npy", oof)
print("OOF saved")


## Пороги и ось

Регрессия по фолдам нужна только чтобы выбрать порог оси.


In [ ]:
for index, column in enumerate(LABEL_COLS):
    result = U.evaluate_oof(
        sp[column].values,
        oof[:, index].reshape(1, -1),
        sp.fold.values.reshape(1, -1),
        sp.study_id.values,
        name=column,
        verbose=True,
    )
    U.update_thresholds({column: result["deployed_threshold"]})

axis_oof = np.full(len(sp), np.nan)
for fold in sorted(sp.fold.unique()):
    train_mask = sp.fold != fold
    val_mask = ~train_mask
    clf = LogisticRegression(class_weight="balanced").fit(
        sp.loc[train_mask, ["axis_angle"]],
        sp.loc[train_mask, "spine_axis"],
    )
    if len(np.unique(sp.loc[val_mask, "spine_axis"])) == 2:
        axis_oof[val_mask] = clf.predict_proba(sp.loc[val_mask, ["axis_angle"]])[:, 1]

axis_eval = U.evaluate_oof(
    sp.spine_axis.values,
    axis_oof.reshape(1, -1),
    sp.fold.values.reshape(1, -1),
    sp.study_id.values,
    name="spine_axis",
    verbose=True,
)
axis_final = LogisticRegression(class_weight="balanced").fit(
    sp[["axis_angle"]],
    sp.spine_axis,
)
joblib.dump(axis_final, "spine_axis_model.joblib")
print(U.update_thresholds({"spine_axis": axis_eval["deployed_threshold"]}))


## Срез spine_lay

Порог развёртывания `0.7592255473136902` жёстче среднего cross-fitted порога `0.40`.

In [ ]:
oof = np.load("spine_cnn_oof.npy")
y = sp.spine_lay.to_numpy()
called = oof[:, 0] >= LAY_THRESHOLD
print(
    "TP",
    int(((called == 1) & (y == 1)).sum()),
    "из",
    int(y.sum()),
    "FP",
    int(((called == 1) & (y == 0)).sum()),
)


## Уже посчитанный результат

```
99 {'spine_lay': 6, 'spine_obj': 17, 'spine_axis': 10}

[spine_lay] n=99, positives=6
  ROC-AUC 0.860 [0.674, 1.000]
  F1 cross-fitted threshold 0.400 [0.000, 0.750]
  sensitivity 0.333 [0.000, 0.800] | specificity 0.978 [0.945, 1.000]
  deployment threshold 0.7592255473136902

[spine_obj] n=99, positives=17
  ROC-AUC 0.781 [0.654, 0.885]
  F1 cross-fitted threshold 0.457 [0.231, 0.642]
  sensitivity 0.471 [0.222, 0.706] | specificity 0.878 [0.807, 0.943]
  deployment threshold 0.4454249441623688

[spine_axis] n=99, positives=10
  ROC-AUC 0.740 [0.539, 0.905]
  F1 cross-fitted threshold 0.261 [0.000, 0.485]
  sensitivity 0.300 [0.000, 0.616] | specificity 0.888 [0.820, 0.946]
  deployment threshold 0.592567792815834

TP 3 из 6 FP 0
```

В `thresholds.json` после этого прогона: `spine_lay` 0.7592255473136902, `spine_obj` 0.4454249441623688, `spine_axis` 0.592567792815834. Ключ `femur_lay` остаётся 0.15252487361431122, инференс бедра его не читает.
